# An Event Study as a Panel Regression

This notebook creates an artificial data set with event---in order to demonstrate how an event study can be implemented as a panel regression with dummies.

In [1]:
MyModulePath = joinpath(pwd(),"src")
!in(MyModulePath,LOAD_PATH) && push!(LOAD_PATH,MyModulePath)
using FinEcmt_OLS

using Dates, DelimitedFiles, Statistics, LinearAlgebra

## Load Data

The data set contains daily data for "dates", the equity market return, riskfree rate and the the returns of the 25 Fama-French portfolios. All returns are in percent.

In [2]:
x   = readdlm("Data/MomentumSR.csv",',')
dN  = Date.(x[:,1],"yyyy-mm-dd")                  #Julia dates
x   = Float64.(x[:,2:end])
(Rm,Rf,R) = (x[:,1],x[:,2],x[:,3:end])

println("size of dN, Rm, Rf, R")
println(size(dN)," ",size(Rm)," ",size(Rf)," ",size(R))

Re  = R .- Rf              #excess returns
Rme = Rm .- Rf             #market excess return
(T,N) = size(Re);

size of dN, Rm, Rf, R
(11598,) (11598,) (11598,) (11598, 25)


## Add artificial Events to the Return Data
  
...to get a new data set that can be used to illustrate an event study.

In [3]:
vv_Events = floor.(Int,range(100,T-100,length=25))    #event indices
σ = std(Re)                                           #magnitude of event effect

ReX = copy(Re)
for i in 1:N
  ReX[vv_Events[i]:vv_Events[i]+1,i] .+= [σ,σ/2]     #add some extra returns
end

## Panel Regression with Individual Dummies

and (market return, cionstant). 

The `EventDummy()` function takes dates or indices to create dummy variables for `p` periods before the event to `q` periods after.

For simplicity, all assets have the same slope coefficients, including the same (α,β) for the "CAPM part." This could easily be changed by, for intance, introdcing `[Rme .* dummy_1,...,Rme .* dummy_N]` as new regressors.

In [4]:
(p,q) = (2,3)                 #dummies for tˣ-p to tˣ+q

D = zeros(Int,T,p+1+q,N)      #create 3D array with dummies
for i in 1:N
  (D[:,:,i],tˣ_i) = EventDummy(vv_Events[i],1:T,p,q)
end

z = repeat([Rme ones(T)],outer=(1,1,N))      #Tx2xN
X = cat(z,D;dims=2)                          #regressors, T x (2+p+1+q) x N

fO     = PanelOls(ReX,X)                     #pooled regression
θ      = fO.theta
StdErr = sqrt.(diag(fO.CovNW))               #White's VCV
tstat  = θ./StdErr

DummyNames = string.("D(",vcat(-p:1:0,1:1:q),")")
rowNames = ["β";"α";DummyNames]
printmat(θ,tstat;rowNames,colNames=["coef","t-stat"])

           coef    t-stat
β         0.979   424.088
α         0.004     3.171
D(-2)     0.129     1.359
D(-1)     0.114     1.559
D(0)      1.333    20.009
D(1)      0.479     4.119
D(2)     -0.042    -0.445
D(3)      0.047     0.414



## Traditional Event Study

where all assets have the same (α,β) in the CAPM regression. This could easily be changed.

In [5]:
fO = PanelOls(ReX,z)                 #pooled CAPM regression, same (α,β) for all
AR = ReX - fO.yhat                   #abnormal return after regressing on (Rme,c)
σ_AR = std(AR)                       #std of AR   

AR_event = fill(NaN,N,p+1+q)         #p periods before, q after
for i in 1:N                         #pick out returns around event daya
  AR_event[i,:] = AR[vv_Events[i]-p:vv_Events[i]+q,i]
end

AR_avg   = mean(AR_event,dims=1)'
AR_tstat = AR_avg./(σ_AR/sqrt(N))

println("Average returns around event days")

printmat(AR_avg,AR_tstat;rowNames=DummyNames,colNames=["coef","t-stat"])

printblue("the average returns are very similar (but not identicial) to the panel regression")

Average returns around event days
           coef    t-stat
D(-2)     0.129     0.993
D(-1)     0.114     0.878
D(0)      1.332    10.287
D(1)      0.479     3.694
D(2)     -0.042    -0.327
D(3)      0.046     0.358

the average returns are very similar (but not identicial) to the panel regression
